In [1]:
%pip install --upgrade onnx onnxscript

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 31.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: onnx
    Found existing installation: onnx 1.20.0
    Uninstalling onnx-1.20.0:
      Successfully uninstalled onnx-1.20.0
Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch
import torch.nn as nn
from torchvision import models

# 1. 모델 구조 선언 (학습 시와 동일한 구조)
num_classes = 3  # Stay, Seat, Lie 등 클래스 개수
device = torch.device("cpu") # export는 CPU에서 하는 것이 가장 안전합니다.

# MobileNetV3 Large 로드 (train.ipynb 기본 설정)
model = models.mobilenet_v3_large(weights=None)
num_ftrs = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(num_ftrs, num_classes)

# 만약 ResNet18을 사용했다면:
# model = models.resnet18(weights=None)
# num_ftrs = model.fc.in_features
# model.fc = nn.Linear(num_ftrs, num_classes)

# 2. 가중치(state_dict) 불러오기
# torch.load('model.pt')가 리턴하는 것은 모델 객체가 아니라 OrderedDict(가중치)입니다.
weights = torch.load('model.pt', map_location=device)
model.load_state_dict(weights)

# 3. 모델을 추론 모드로 전환 (필수!)
model.eval()

# 4. 더미 입력 생성 (모델 입력 사이즈 확인: 224x224)
dummy_input = torch.randn(1, 3, 224, 224, device=device)

# 5. ONNX로 내보내기
onnx_file_path = "model.onnx"
torch.onnx.export(
    model,                  # 이제 '가중치'가 아니라 '모델 객체'를 넘깁니다.
    dummy_input,            
    onnx_file_path,         
    export_params=True,     
    opset_version=12,       
    do_constant_folding=True,
    input_names=['input'],   
    output_names=['output']
)

print(f"성공적으로 변환되었습니다: {onnx_file_path}")

성공적으로 변환되었습니다: model.onnx
